In [1]:
import numpy as np
import pandas as pd
import math
from scipy.stats import norm
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'numpy'

In [ ]:
def vorst_algo(u, d, R, S0, K, n, k):
    """
    Calcule le coût de réplication d'un Short Call (Palmer 2001).
    Renvoie: (coût_initial, arbre_des_deltas)
    """

    S_tree = [[0.0 for j in range(i + 1)] for i in range(n + 1)]
    Delta_tree = [[0.0 for j in range(i + 1)] for i in range(n + 1)]
    B_tree = [[0.0 for j in range(i + 1)] for i in range(n + 1)]

    for i in range(n + 1):
        for j in range(i + 1):
            S_tree[i][j] = S0 * (u**j) * (d**(i - j))

    for j in range(n + 1):
        if S_tree[n][j] > K:
            Delta_tree[n][j] = -1.0
            B_tree[n][j] = K
        else:
            Delta_tree[n][j] = 0.0
            B_tree[n][j] = 0.0

    
    for i in range(n - 1, -1, -1):
        for j in range(i + 1):
            S_node = S_tree[i][j]
            Delta_u, B_u = Delta_tree[i + 1][j + 1], B_tree[i + 1][j + 1]
            Delta_d, B_d = Delta_tree[i + 1][j], B_tree[i + 1][j]

            C0 = (Delta_u * u - Delta_d * d) / (u - d) + (B_u - B_d) / (S_node * (u - d))
            
            def f(delta):
                return delta - C0 - (k / (u - d)) * (abs(Delta_u - delta) * u - abs(Delta_d - delta) * d)

            S_u_sign = 1.0 if f(Delta_u) >= 0 else -1.0
            S_d_sign = 1.0 if f(Delta_d) >= 0 else -1.0

            num = C0 + (k / (u - d)) * (S_u_sign * Delta_u * u - S_d_sign * Delta_d * d)
            den = 1.0 + (k / (u - d)) * (S_u_sign * u - S_d_sign * d)
            Delta_current = num / den

            B_current = (Delta_u * S_node * u + B_u + k * abs(Delta_u - Delta_current) * S_node * u - Delta_current * S_node * u) / R

            Delta_tree[i][j] = Delta_current
            B_tree[i][j] = B_current

    cost = - (Delta_tree[0][0] * S0 + B_tree[0][0])
    return cost, Delta_tree

# --- Variables initiales ---
S0 = 100.0
sigma = 0.20
r = math.log(1.1)
T = 1.0

ns = [6, 13, 52]
Ks = [80, 90, 100, 110, 120]
ks = [0, 0.00125, 0.005, 0.02]

results = []
for k in ks:
    for K in Ks:
        row = {'k': k, 'K': K}
        for n in ns:
            # Paramètre du papier
            dt = T / n
            u = math.exp(sigma * math.sqrt(dt))
            d = 1.0 / u
            R = math.exp(r * dt)
            
            cost, _ = vorst_algo(u, d, R, S0, K, n, k)
            row[f'n={n}'] = round(cost, 3)
        results.append(row)

df = pd.DataFrame(results)
print(df.to_string(index=False))
#PS: on a le même résultat que sur la note de Boyle et Vorst par Palmer

      k   K    n=6   n=13      n=52
0.00000  80 27.703 27.701    27.665
0.00000  90 19.821 19.740    19.667
0.00000 100 12.655 13.093    12.953
0.00000 110  8.129  8.026     7.972
0.00000 120  4.216  4.427     4.548
0.00125  80 27.671 27.656    27.582
0.00125  90 19.749 19.638    19.469
0.00125 100 12.538 12.935    12.638
0.00125 110  8.003  7.843     7.604
0.00125 120  4.102  4.256     4.202
0.00500  80 27.582 27.534    27.383
0.00500  90 19.531 19.333    18.889
0.00500 100 12.177 12.445    11.606
0.00500 110  7.614  7.269     6.374
0.00500 120  3.754  3.726     3.077
0.02000  80 27.327 27.276   -92.878
0.02000  90 18.697 18.281  -652.937
0.02000 100 10.561 10.115 -2121.964
0.02000 110  5.845  4.311  -849.031
0.02000 120  2.266  1.266  -362.472


In [ ]:
def bv_long_call(S0, K, T, sigma, r_eff, n, k):
    """
    Calcule le coût de réplication d'un Long Call selon Boyle et Vorst (1992).
    Utilise une approche vectorisée avec une complexité spatiale O(n).
    """
    dt = T / n
    u = math.exp(sigma * math.sqrt(dt))
    d = 1 / u
    R = (1 + r_eff) ** dt 
    
    # Construction du vecteur des prix terminaux (à maturité)
    S_T = S0 * (u ** np.arange(n, -1, -1)) * (d ** np.arange(0, n + 1))
    
    # Payoffs terminaux de l'option d'achat
    Delta = np.where(S_T > K, 1.0, 0.0)
    B = np.where(S_T > K, -K, 0.0)
    
    u_bar = u * (1 + k)
    d_bar = d * (1 - k)
    
    # Rétro-induction
    for i in range(n - 1, -1, -1):
        # Prix de l'actif sous-jacent au nœud i
        S_nodes = S0 * (u ** np.arange(i, -1, -1)) * (d ** np.arange(0, i + 1))
        
        # Portefeuilles aux nœuds fils (up et down)
        Delta_u = Delta[:-1]
        B_u = B[:-1]
        Delta_d = Delta[1:]
        B_d = B[1:]
        
        # Valeur du portefeuille nécessaire dans chaque état (incluant les frais de transaction virtuels)
        V_u = Delta_u * S_nodes * u_bar + B_u
        V_d = Delta_d * S_nodes * d_bar + B_d
        
        # Résolution du système linéaire
        Delta = (V_u - V_d) / (S_nodes * (u_bar - d_bar))
        B = (V_u - Delta * S_nodes * u_bar) / R
        
    return Delta[0] * S0 + B[0]

# --- 1. Validation de la Table I ---
S0 = 100
sigma = 0.2
T = 1.0
r_eff = 0.1

strikes = [80, 90, 100, 110, 120]
ns = [6, 13, 52]
ks = [0, 0.00125, 0.005, 0.02]

res = []
for k in ks:
    for K in strikes:
        row = {'k': k, 'K': K}
        for n in ns:
            row[f'n={n}'] = bv_long_call(S0, K, T, sigma, r_eff, n, k)
        res.append(row)

df_table1 = pd.DataFrame(res)
print("Table I Long Call Prices:")
print(df_table1.round(3).to_string(index=False))


# --- 2. Convergence vers Black-Scholes modifié en fonction de n ---
def bs_call(S0, K, T, r_c, sigma):
    """
    Formule classique de Black-Scholes pour un Call européen.
    """
    if sigma <= 0: return max(S0 - K*math.exp(-r_c*T), 0)
    d1 = (math.log(S0 / K) + (r_c + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S0 * norm.cdf(d1) - K * math.exp(-r_c * T) * norm.cdf(d2)

# Conversion du taux effectif annuel en taux continu
r_c = math.log(1 + r_eff)
k_fixed = 0.01
K_fixed = 100
n_vals = np.arange(10, 251, 10)

bv_prices_n = []
mod_bs_prices_n = []

for n in n_vals:
    # Prix discret
    bv_p = bv_long_call(S0, K_fixed, T, sigma, r_eff, n, k_fixed)
    bv_prices_n.append(bv_p)
    
    # Volatilité modifiée de Boyle et Vorst
    mod_vol = sigma * math.sqrt(1 + (2 * k_fixed * math.sqrt(n)) / (sigma * math.sqrt(T)))
    # Prix Black-Scholes avec volatilité modifiée
    mod_bs_p = bs_call(S0, K_fixed, T, r_c, mod_vol)
    mod_bs_prices_n.append(mod_bs_p)
    

In [ ]:

plt.figure(figsize=(10,6))
plt.plot(n_vals, bv_prices_n, label='Modele Discret de Boyle-Vorst', marker='o', markersize=3)
plt.plot(n_vals, mod_bs_prices_n, label='Approximation de Black-Scholes Modifiee', linestyle='--')
plt.title("Convergence vers BS modifie en fonction du nombre de pas n")
plt.xlabel("Nombre de pas de discretisation (n)")
plt.ylabel("Prix de l'option Long Call")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# --- 3. Convergence vers le modèle sans friction lorsque k tend vers 0 ---
n_fixed = 50
k_vals = np.linspace(0, 0.05, 50)
bv_prices_k = []
bs_standard = bs_call(S0, K_fixed, T, r_c, sigma)

for k_ in k_vals:
    bv_prices_k.append(bv_long_call(S0, K_fixed, T, sigma, r_eff, n_fixed, k_))
    
plt.figure(figsize=(10,6))
plt.plot(k_vals, bv_prices_k, label=f'Modele Boyle-Vorst (n={n_fixed})', marker='x')
plt.axhline(bs_standard, color='r', linestyle='--', label='Black-Scholes (sans friction)')
plt.title("Convergence vers le modele sans friction lorsque k tend vers 0")
plt.xlabel("Cout de transaction (k)")
plt.ylabel("Prix de l'option Long Call")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()